<a href="https://colab.research.google.com/github/lhammach/DP-representations/blob/main/cka_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CKA analysis of ResNet18 representations


This notebook compares internal representations learned by:

- Baseline ResNet18
- DP-SGD ResNet18

using Centered Kernel Alignment (CKA).

The models were trained separately and saved as checkpoints.

## 0. Setup

In [ ]:
import torch
import torchvision.transforms as transforms

from torchvision.datasets import ImageFolder
from torchvision import models

import torch.nn as nn

import numpy as np
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm

## 1. Dataset

We load CIFAR10 exactly as during training.

Using the same normalization is critical because activations depend on input scaling.

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD_DEV = (0.2023, 0.1994, 0.2010)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD_DEV),
])

test_dataset = ImageFolder(
    "./cifar10/test",
    transform=transform,
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
)

## 2. Rebuild models

We recreate the exact architecture used during training
and load saved weights.

### 2.1 Baseline

In [ ]:
baseline = models.resnet18(num_classes=10)

### 2.2 DP model

We do exactly the same corrections as in the notebook where the models are trained.

In [ ]:
dp_model = models.resnet18(num_classes=10)

# corrections

from torchvision.models.resnet import BasicBlock

def safe_forward(self, x):
    identity = x

    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)

    out = self.conv2(out)
    out = self.bn2(out)

    if self.downsample is not None:
        identity = self.downsample(x)

    out = out + identity
    out = self.relu(out)

    return out

BasicBlock.forward = safe_forward

# desactivate inplace ReLU

for module in dp_model.modules():
    if isinstance(module, nn.ReLU):
        module.inplace = False

# Conversion BatchNorm -> GroupNorm

from opacus.validators import ModuleValidator

dp_model = ModuleValidator.fix(dp_model)

### 2.3 Load checkpoints

In [ ]:
baseline.load_state_dict(
    torch.load("baseline_resnet18.pth", map_location="cpu")
)

dp_model.load_state_dict(
    torch.load("dp_resnet18_eps50.pth", map_location="cpu")
)

# GPU

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

baseline.to(device)
dp_model.to(device)

baseline.eval()
dp_model.eval()

## 3. Extract representations

In [ ]:
layers = [
    "layer1",
    "layer2",
    "layer3",
    "layer4",
]

### 3.1 Hook utility

In [ ]:
def get_activations(model, layer_names):
    activations = {}

    def hook_fn(name):
        def hook(module, inp, out):
            activations[name] = out.detach()
        return hook

    hooks = []

    for name, module in model.named_modules():
        if name in layer_names:
            hooks.append(
                module.register_forward_hook(
                    hook_fn(name)
                )
            )

    return activations, hooks

### 3.2 Run one batch

In [ ]:
images, labels = next(iter(test_loader))
images = images.to(device)

acts_base, hooks_base = get_activations(
    baseline,
    layers,
)

acts_dp, hooks_dp = get_activations(
    dp_model,
    layers,
)

with torch.no_grad():
    baseline(images)
    dp_model(images)

# withdraw hooks

for h in hooks_base:
    h.remove()

for h in hooks_dp:
    h.remove()

## 4. CKA

We use linear CKA as proposed by Kornblith et al.

### 4.1 HSIC

In [ ]:
def gram_linear(x):
    return x @ x.T

def center_gram(K):
    n = K.shape[0]

    H = torch.eye(n, device=K.device)
    H -= torch.ones(n, n, device=K.device) / n

    return H @ K @ H

def hsic(K, L):
    return torch.sum(K * L)

### 4.2 Linear CKA

In [ ]:
def linear_cka(X, Y):

    X = X.flatten(1)
    Y = Y.flatten(1)

    K = center_gram(
        gram_linear(X)
    )

    L = center_gram(
        gram_linear(Y)
    )

    hsic_xy = hsic(K, L)

    hsic_xx = hsic(K, K)
    hsic_yy = hsic(L, L)

    return (
        hsic_xy /
        torch.sqrt(hsic_xx * hsic_yy)
    ).item()

## 5. Compute CKA

Similarity by layer.

In [ ]:
cka_scores = {}

for layer in layers:

    score = linear_cka(
        acts_base[layer],
        acts_dp[layer],
    )

    cka_scores[layer] = score

    print(layer, score)

## 6. Plot

Similarity across layers.

In [ ]:
plt.figure(figsize=(6,4))

plt.bar(
    list(cka_scores.keys()),
    list(cka_scores.values()),
)

plt.ylabel("CKA")
plt.ylim(0,1)

plt.show()